<a href="https://colab.research.google.com/github/julschleinitz/ai4chemistry-bootcamp/blob/main/tutorials/02-molecular-representations/pka-data-generation.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generating `esol_pka.csv` with pkasolver

This is a **data-generation notebook**, not a student exercise: it predicts a pKa value (and an
`is_ionizable` flag) for every molecule in the ESOL solubility dataset using
[**pkasolver**](https://github.com/mayrf/pkasolver) (Mayr, Wieder, Wieder & Langer — a graph neural
network ensemble trained to predict microstate pKa values), and writes the result to
`tutorials/data/esol_pka.csv`. That file is read by `merge_pka` in `molecular-representations.ipynb` /
`molecular-representations_solutions.ipynb` to build the charge/pKa descriptor block — this notebook is
how the instructor (re)generates it; students don't need to run this.

## Setup

**This notebook needs its own environment, separate from `environment.yml`.** pkasolver pins
`torch-geometric==2.0.1` plus the older `torch-scatter`/`torch-sparse` extensions it depends on — and
that's incompatible with the modern `torch-geometric` (and the `deepchem` version built against it) used
by the two main tutorial notebooks. Mixing them in one environment breaks one side or the other, so this
notebook is designed to be run in its own isolated environment.

**Local setup:**
```bash
conda create -n pkasolver-env python=3.9 -y
conda install -n pkasolver-env -c conda-forge -c pytorch rdkit pytorch cpuonly numpy scipy tqdm svgutils cairosvg pip -y
conda run -n pkasolver-env pip install torch-geometric==2.0.1 torch-scatter torch-sparse molvs pandas "git+https://github.com/mayrf/pkasolver.git"
```
(`torch-scatter`/`torch-sparse` compile from source — a couple of minutes, not pre-built wheels for every
platform.) A matching `environment_pkasolver.yml` is included alongside this notebook for `conda env
create -f environment_pkasolver.yml`.

**Colab:** run the cell below instead (same packages via pip; skip the conda block above).

In [ ]:
# Colab-only install (skip if using the local pkasolver-env conda environment above)
# !pip install -q rdkit torch pandas tqdm
# !pip install -q torch-geometric==2.0.1 torch-scatter torch-sparse molvs
# !pip install -q "git+https://github.com/mayrf/pkasolver.git"

In [ ]:
import contextlib
import io
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')  # quiet RDKit's routine parsing warnings

from pkasolver.query import calculate_microstate_pka_values, QueryModel

print("Imports OK")

## Load the full ESOL dataset

This notebook avoids importing `deepchem` directly (its `torch_models` submodule requires the same modern
`torch-geometric` this environment deliberately doesn't have) — instead we read the same underlying CSV
`dc.molnet.load_delaney` downloads under the hood, so the SMILES/order match the main tutorial notebooks
exactly.

In [ ]:
ESOL_URL = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv'
df_all = pd.read_csv(ESOL_URL)[['smiles', 'measured log solubility in mols per litre']]
df_all.columns = ['smiles', 'logS']

print(f"Loaded {len(df_all)} molecules")
df_all.head()

## Sanity check on molecules with known pKa values

Before running the full dataset, check pkasolver against a few textbook pKa values: acetic acid (~4.76),
phenol (~10.0), aniline (~4.6, protonation of the amine), and hexane (no ionizable group at all).

**Note on verbose output:** `calculate_microstate_pka_values` shells out to dimorphite-dl as a subprocess
per ionizable site, which prints a citation notice to the terminal every time — that output is inherited
directly by the OS-level stdout, so it isn't caught by `contextlib.redirect_stdout` and will show up below
regardless (confirmed: an `os.dup2`-based fd redirect doesn't reliably suppress it either once running
inside a Jupyter kernel, since ipykernel forwards child-process output through its own pipe). It's harmless
— just collapse the cell output in Jupyter/Colab if it's distracting.

In [ ]:
query_model = QueryModel()  # loads the 25-model ensemble bundled with the package

known = {
    'CC(=O)O':       ('acetic acid', 4.76),
    'c1ccc(cc1)O':   ('phenol', 10.0),
    'c1ccccc1N':     ('aniline', 4.6),
    'CCCCCC':        ('hexane', None),
}

for smi, (name, lit_pka) in known.items():
    mol = Chem.MolFromSmiles(smi)
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        states = calculate_microstate_pka_values(mol, query_model=query_model)
    pkas = [f'{s.pka:.2f}' for s in states]
    print(f"{name:10s} (lit. pKa {lit_pka}): predicted {pkas if pkas else 'no ionizable group found'}")

## Predict pKa for the full dataset

Each molecule may have several ionizable sites (pkasolver enumerates them all); we keep the single pKa
**closest to pH 7**, since that's the one most relevant to whether the molecule is significantly ionized
under the near-neutral aqueous conditions ESOL's solubility was measured in. `is_ionizable` is 1 if
pkasolver found *any* ionizable site, 0 otherwise (e.g. plain alkanes like hexane above).

In [ ]:
def predict_pka(smi, query_model):
    """Predict the pKa closest to pH 7 for one SMILES string.

    Returns a dict with `pka` (None if no ionizable group or on failure), `is_ionizable`,
    `n_states` (number of ionizable sites pkasolver found), and `error` (None on success).
    """
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return {'pka': None, 'is_ionizable': 0, 'n_states': 0, 'error': 'invalid SMILES'}

    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            states = calculate_microstate_pka_values(mol, query_model=query_model)
    except Exception as e:
        return {'pka': None, 'is_ionizable': 0, 'n_states': 0, 'error': str(e)}

    if not states:
        return {'pka': None, 'is_ionizable': 0, 'n_states': 0, 'error': None}

    best = min(states, key=lambda s: abs(s.pka - 7.0))
    return {'pka': float(best.pka), 'is_ionizable': 1, 'n_states': len(states), 'error': None}

In [ ]:
t0 = time.time()
results = []
for smi in tqdm(df_all['smiles']):
    results.append(predict_pka(smi, query_model))
elapsed = time.time() - t0

n_failed = sum(1 for r in results if r['error'] is not None)
n_ionizable = sum(r['is_ionizable'] for r in results)
print(f"Done in {elapsed/60:.1f} min ({elapsed/len(df_all):.2f}s/molecule)")
print(f"{n_ionizable}/{len(df_all)} molecules flagged ionizable, {n_failed} failed (kept as pka=NaN, is_ionizable=0)")

## Save `esol_pka.csv`

Same schema `merge_pka` (in the main tutorial notebooks) expects: `smiles, pka, is_ionizable` — keyed on
the *original* SMILES here, since `merge_pka` canonicalizes both sides at merge time anyway.

In [ ]:
pka_df = pd.DataFrame({
    'smiles': df_all['smiles'],
    'pka': [r['pka'] for r in results],
    'is_ionizable': [r['is_ionizable'] for r in results],
})

output_path = 'data/esol_pka.csv'
pka_df.to_csv(output_path, index=False)
print(f"Wrote {len(pka_df)} rows to {output_path}")
pka_df.head()

## Caveats

- **Single pKa per molecule.** Molecules with several ionizable groups (e.g. amino acids) are collapsed
  to just the one closest to pH 7 — enough for the "is this molecule likely charged near neutral pH?"
  descriptor used in the main tutorial, but it discards the rest of pkasolver's microstate output
  (`n_states` above; each `State` also carries `.pka_stddev`, the ensemble's disagreement, dropped here).
- **Accuracy varies by chemistry.** pkasolver-lite (the model shipped here, trained without the
  transfer-learning step described in the paper) is trained mostly on drug-like molecules. Simple
  aliphatic alcohols in particular came out noticeably off in spot checks above and during development
  (predicted ~9-10 vs. a literature pKa around 16 for a plain alcohol) — treat pKa values for
  non-drug-like ESOL entries (alkanes, simple alcohols, sugars) as rougher estimates than for
  pharmaceutical-like molecules.
- **Failures are kept, not dropped.** Any molecule pkasolver couldn't process keeps its row with
  `pka=NaN, is_ionizable=0`, mirroring the "keep as NaN, don't drop the molecule" convention `merge_pka`
  already expects (rather than silently shrinking the dataset).

**Citation:** Mayr, F., Wieder, M., Wieder, O., Langer, T. *Improving Small Molecule pKa Prediction Using
Transfer Learning with Graph Neural Networks.* [bioRxiv 2022.01.20.476787](https://www.biorxiv.org/content/10.1101/2022.01.20.476787v1).
Also bundles [Dimorphite-DL](https://github.com/durrantlab/dimorphite_dl) (Ropp et al., *J Cheminform* 2019)
for protonation-site enumeration.